In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets,transforms,models
import time
import copy
def build_transfer_model(num_classes:int,freeze_backbone:bool = True) -> nn.Module:
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
        print(f"[INFO] 骨干网络已冻结，共 {sum(p.numel() for p in model.parameters())} 个参数被锁定")
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(num_features,256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256,num_classes)
    )
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"[INFO] 可训练参数: {trainable_params:,} / {total_params:,} "
          f"({100 * trainable_params / total_params:.2f}%)")
    return model
def get_dataloader(batch_size:int=128,data_dir:str='./data'):
    imagenet_mean = [0.485,0.456,0.406]
    imagenet_std = [0.229,0.224,0.225]
    transform_train = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomResizedCrop(224),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean,imagenet_std),
    ])
    transform_val = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean,imagenet_std),
    ])
    train_dataset = datasets.MNIST(root=data_dir,train=True,download=True,transform=transform_train)
    val_dataset = datasets.MNIST(root = data_dir,train=False,download=True,transform=transform_val)
    train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True,num_workers=2,pin_memory=True)
    val_loader = DataLoader(val_dataset,batch_size=batch_size,shuffle=True,num_workers=2,pin_memory=True)
    print(f"[MNIST] 训练集: {len(train_dataset):,} 张 | 验证集: {len(val_dataset):,} 张")
    return train_loader, val_loader
def train_one_epoch(model,loader,criterion,optimizer,device):
    model.train()
    running_loss = 0.0
    correct,total = 0,0
    for images,labels in loader:
        images,labels = images.to(device),labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        running_loss = loss.item() * images.size(0)
        _,preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total
def evaluate(model,loader,criterion,device):
    model.eval()
    running_loss,correct,total = 0.0,0,0
    for images,labels in loader:
        images,labels = images.to(device),labels.to(device)
        outputs = model(images)
        loss = criterion(outputs,labels)
        running_loss = loss.item() * images.size(0)
        _,preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total,correct/total
def run_training(model,train_loader,val_loader,device,epochs=10,lr=1e-3,dataset_name='Dataset'):
    optimizer = optim.Adam(
        filter(lambda p:p.requires_grad,model.parameters()),
        lr=lr,weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    best_acc = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    print(f"\n{'='*60}")
    print(f" 开始训练 [{dataset_name}] — 共 {epochs} 个 Epoch")
    print(f"{'='*60}")
    for epoch in range(1,epochs+1):
        start_time = time.time()
        train_loss,train_acc = train_one_epoch(model,train_loader,criterion,optimizer,device)
        val_loss,val_acc = evaluate(model,val_loader,criterion,device)
        scheduler.step()
        elapsed = time.time() - start_time
        lr_now = optimizer.param_groups[0]['lr']
        print(f"  Epoch [{epoch:02d}/{epochs}]  "
              f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} | "
              f"LR: {lr_now:.6f} | {elapsed:.1f}s")
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_model_wts)
    save_path = f"{dataset_name}_best.pth"
    torch.save(best_model_wts, save_path)
    print(f"\n✅ [{dataset_name}] 训练完成！最佳验证准确率: {best_acc:.4f}")
    return model